# 🤖 Agente RAG sobre Sales Dataset — Dataset Normalizado
## Objetivo: construir un bot que busque, filtre y responda preguntas sobre el corpus de ventas

> **El RAG debe ser capaz de:**
> - Ir al corpus (dataset de ventas) y recuperar los registros relevantes
> - Filtrar por producto, país, cliente, estado, año, etc.
> - Decirte **exactamente dónde está** un producto u orden (ciudad, país, dirección)
> - Responder preguntas libres en lenguaje natural, fundamentado en los datos reales

---
### 📋 Diferencias con la versión original
Este notebook usa el **dataset ya normalizado** (`sales_data_normalized.csv`), que incluye:
- Columnas en `snake_case` minúsculas
- `orderdate` como fecha real (ISO 8601)
- Nulos tratados (`addressline2`, `state`, `postalcode`, `territory`)
- Columna ordinal `dealsize_num` (Small=1, Medium=2, Large=3)
- Sin columnas redundantes (`qtr_id`, `month_id`, `year_id` eliminadas)

**El paso 6 (normalización) no es necesario — el dato ya llega limpio.**

---
### 👨‍🏫 Stack tecnológico
| Componente | Herramienta |
|---|---|
| LLM | **Mistral AI** (mistral-small-latest) |
| Embeddings | sentence-transformers (local, gratis) |
| Vector DB | **FAISS** (local, sin servidor) |
| Framework RAG | **LangChain** |
| Data wrangling | **Pandas** |
| Secrets | **Google Colab Secrets** (forma segura) |

---
## 🗺️ Ruta de Trabajo

```
FASE 1 — SETUP (15 min)
  ├── Paso 1 │ Configurar API Key en Colab Secrets   (5 min)
  ├── Paso 2 │ Instalar dependencias                 (5 min)
  └── Paso 3 │ Importar librerías y verificar        (2 min)

FASE 2 — DATA (15 min)
  ├── Paso 4 │ Cargar dataset normalizado             (3 min)
  ├── Paso 5 │ Exploración EDA con Pandas            (7 min)
  └── Paso 6 │ Enriquecer con columnas derivadas     (5 min)

FASE 3 — RAG PIPELINE (20 min)
  ├── Paso 7 │ Convertir filas → Documentos RAG      (5 min)
  ├── Paso 8 │ Embeddings + índice FAISS             (7 min)
  └── Paso 9 │ Prompt + Cadena RAG con Mistral       (8 min)

FASE 4 — BOT (10 min)
  ├── Paso 10│ Probar con preguntas de ejemplo       (5 min)
  └── Paso 11│ Bot interactivo libre                 (5 min)
```

---
## 📚 Conceptos Clave

### ¿Qué es RAG?
**RAG (Retrieval-Augmented Generation)** combina dos etapas:

```
PREGUNTA DEL USUARIO
        │
        ▼
┌───────────────────┐
│  RETRIEVAL        │  ← El corpus (dataset) se busca semánticamente
│  FAISS + Embeds   │    y se extraen los fragmentos más relevantes
└───────────────────┘
        │
        ▼
┌───────────────────┐
│  GENERATION       │  ← El LLM (Mistral) recibe esos fragmentos
│  Mistral LLM      │    como contexto y genera la respuesta
└───────────────────┘
        │
        ▼
RESPUESTA BASADA EN TUS DATOS REALES
```

### ¿Por qué RAG sobre un dataset tabular?
- El LLM solo conoce hasta su fecha de corte → **no conoce tu dataset**
- RAG le inyecta los datos relevantes en cada pregunta
- El bot puede decirte: *"El producto S10_1678 está en NYC, USA, en 897 Long Airport Avenue"*

### ¿Qué es un Agente?
Un agente puede usar **herramientas** (tools). En esta clase el agente tiene:
- 🔧 `buscar_en_corpus` — búsqueda semántica libre
- 🔧 `localizar_producto` — localiza dónde está un producto
- 🔧 `filtrar_por_pais` — filtra por país
- 🔧 `filtrar_por_producto` — filtra por línea de producto
- 🔧 `resumen_estadistico` — estadísticas del dataset

---
## 🔐 PASO 1 — Configurar API Keys (Forma Segura con Colab Secrets)

### ¿Cómo agregar tu API Key de Mistral?
1. Ir a [https://console.mistral.ai](https://console.mistral.ai) → **API Keys** → crear una
2. En Colab: click en el ícono **🔑 (llave)** del panel izquierdo
3. Click **"+ Add new secret"**
4. Nombre: `MISTRAL_API_KEY` — Valor: tu API Key
5. Activar el toggle **"Notebook access"** ✅

> ⛔ **NUNCA** escribas la API Key directamente en el código.  
> ✅ **SIEMPRE** usá Colab Secrets — el token no queda en el historial ni en el archivo.

In [ ]:
# ─── Cargar API Key desde Colab Secrets ─────────────────────────
from google.colab import userdata
import os

try:
    MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
    os.environ["MISTRAL_API_KEY"] = MISTRAL_API_KEY
    masked = MISTRAL_API_KEY[:6] + "..." + MISTRAL_API_KEY[-4:]
    print(f"✅ MISTRAL_API_KEY cargada correctamente: {masked}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("→ Agregá 'MISTRAL_API_KEY' en Colab Secrets (ícono 🔑 del panel izquierdo)")
    raise

---
## 📦 PASO 2 — Instalar Dependencias

| Paquete | Para qué sirve |
|---|---|
| `langchain` | Framework principal para RAG y agentes |
| `langchain-mistralai` | Conector oficial Mistral ↔ LangChain |
| `langchain-community` | FAISS, HuggingFace embeddings, tools |
| `faiss-cpu` | Base de datos vectorial local (ultrarrápida) |
| `sentence-transformers` | Modelo de embeddings multilingüe (gratis, local) |
| `pandas` | Carga y análisis del dataset |
| `numpy` | Operaciones numéricas |
| `tabulate` | Mostrar tablas en consola |

In [ ]:
# ─── Instalar todo el stack RAG ──────────────────────────────────
print("⏳ Instalando dependencias (puede tardar 2-3 minutos)...")

!pip install -q \
    langchain==0.3.7 \
    langchain-mistralai \
    langchain-community \
    faiss-cpu \
    sentence-transformers \
    pandas \
    numpy \
    tabulate

print("\n✅ Todas las dependencias instaladas correctamente")

---
## 🐍 PASO 3 — Importar Librerías y Verificar Versiones

In [ ]:
# ─── Imports principales ─────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.tools import Tool
from langchain.agents import initialize_agent, AgentType

# Utils
from tabulate import tabulate
from IPython.display import display, HTML

print("✅ Librerías importadas correctamente")
print(f"   pandas     : {pd.__version__}")
print(f"   numpy      : {np.__version__}")
print("   langchain  : OK")
print("   mistralai  : OK")
print("   faiss-cpu  : OK")

---
## 🗃️ PASO 4 — Cargar el Dataset Normalizado

### Sobre el dataset: `sales_data_normalized.csv`
Este archivo ya fue normalizado previamente. Sus características:
- **2823 órdenes** de ventas globales
- **23 columnas** en `snake_case` (sin redundancias)
- Columnas de texto limpias, nulos tratados, `orderdate` en ISO 8601
- Columna `dealsize_num`: Small=1, Medium=2, Large=3
- Rango temporal: 2003–2005 | 19 países

### Subir el archivo a Colab:
**Opción A** — Subida manual (ejecutá la celda de carga)  
**Opción B** — Desde Google Drive (montalo primero)

In [ ]:
# ─── Opción A: Subida manual desde tu PC ────────────────────────
# Descomentá si necesitás subir el archivo:
# from google.colab import files
# uploaded = files.upload()  # seleccioná sales_data_normalized.csv

# ─── Opción B: Desde Google Drive ───────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/sales_data_normalized.csv'

# ─── Cargar el CSV normalizado ───────────────────────────────────
CSV_PATH = 'sales_data_normalized.csv'  # ajustá si la ruta es distinta

df = pd.read_csv(CSV_PATH, parse_dates=['orderdate'])

print(f"✅ Dataset normalizado cargado")
print(f"   Filas    : {df.shape[0]:,}")
print(f"   Columnas : {df.shape[1]}")
print(f"   Nulos    : {df.isnull().sum().sum()} (esperado: algunos en addressline2)")
print(f"\n📋 Columnas disponibles:")
print(df.columns.tolist())
print(f"\n📋 Primeras 3 filas:")
display(df.head(3))

---
## 🐼 PASO 5 — Exploración con Pandas (EDA)

El dataset ya llega limpio, pero igual exploramos para entender el corpus antes de construir el RAG.

In [ ]:
# ═══ EDA COMPLETO ════════════════════════════════════════════════

# ── 1. Tipos de datos y nulos ─────────────────────────────────────
print("=" * 60)
print("📋 TIPOS DE DATOS Y VALORES NULOS")
print("=" * 60)
info_df = pd.DataFrame({
    'Tipo': df.dtypes,
    'No Nulos': df.count(),
    'Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(1)
})
print(tabulate(info_df, headers='keys', tablefmt='grid'))

# ── 2. Estadísticas numéricas ─────────────────────────────────────
print("\n" + "=" * 60)
print("📈 ESTADÍSTICAS — COLUMNAS NUMÉRICAS CLAVE")
print("=" * 60)
num_stats = df[['quantityordered','priceeach','sales','msrp','dealsize_num']].describe().round(2)
display(num_stats)

# ── 3. Valores únicos categóricos ────────────────────────────────
print("\n" + "=" * 60)
print("🏷️  DISTRIBUCIÓN DE VARIABLES CATEGÓRICAS")
print("=" * 60)

for col in ['status', 'productline', 'dealsize', 'territory']:
    vc = df[col].value_counts()
    print(f"\n🔹 {col}:")
    for val, cnt in vc.items():
        bar = '█' * int(cnt / 50)
        print(f"   {str(val):<25} {cnt:>4} {bar}")

# ── 4. Países con más órdenes ─────────────────────────────────────
print("\n" + "=" * 60)
print("🌎 TOP 10 PAÍSES POR NÚMERO DE ÓRDENES")
print("=" * 60)
country_top = df['country'].value_counts().head(10)
for country, cnt in country_top.items():
    bar = '█' * int(cnt / 20)
    print(f"   {country:<20} {cnt:>4} {bar}")

# ── 5. Ventas por año ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("📅 VENTAS TOTALES POR AÑO")
print("=" * 60)
df['order_year'] = df['orderdate'].dt.year
yr_sales = df.groupby('order_year')['sales'].agg(['sum','count','mean']).round(2)
yr_sales.columns = ['Total $', 'N Órdenes', 'Promedio $']
print(tabulate(yr_sales, headers='keys', tablefmt='grid'))

---
## 🔧 PASO 6 — Enriquecer con Columnas Derivadas

El dataset normalizado ya está limpio. Acá agregamos columnas calculadas que **enriquecen el texto del corpus RAG**:
- `revenue_calc` — cantidad × precio (validación cruzada con `sales`)
- `discount_pct` — descuento respecto al MSRP
- `contact_full` — nombre completo del contacto
- `full_address` — dirección completa en una sola línea
- `order_quarter` — trimestre derivado de la fecha

> ⚠️ **Diferencia clave con la versión original:** aquí NO hay que parsear fechas ni rellenar nulos — eso ya lo hace el dataset. Solo agregamos derivadas para enriquecer el RAG.

In [ ]:
# ═══ COLUMNAS DERIVADAS PARA ENRIQUECER EL RAG ═══════════════════

# Columnas derivadas de fecha
df['order_year']    = df['orderdate'].dt.year.astype('Int64')
df['order_month']   = df['orderdate'].dt.month.astype('Int64')
df['order_quarter'] = df['orderdate'].dt.quarter.astype('Int64')

# Columnas derivadas de negocio
df['revenue_calc'] = (df['quantityordered'] * df['priceeach']).round(2)
df['discount_pct'] = ((1 - df['priceeach'] / df['msrp']) * 100).clip(0).round(1)
df['contact_full'] = (df['contactfirstname'].fillna('') + ' ' + df['contactlastname'].fillna('')).str.strip()

# Dirección completa en una sola línea
df['addressline2_clean'] = df['addressline2'].apply(
    lambda x: f', {x}' if pd.notna(x) and str(x).strip() not in ['', 'nan'] else ''
)
df['full_address'] = (
    df['addressline1'].fillna('').str.strip()
    + df['addressline2_clean']
    + ', ' + df['city'].fillna('')
    + df['state'].apply(lambda x: f', {x}' if pd.notna(x) and x not in ['N/A', 'nan'] else '')
    + ', ' + df['country'].fillna('')
)
df.drop(columns=['addressline2_clean'], inplace=True)

# Tipos correctos
df['ordernumber'] = df['ordernumber'].astype(str)

print("✅ COLUMNAS DERIVADAS AGREGADAS")
print(f"   Total columnas : {len(df.columns)}")
nuevas = ['order_year', 'order_month', 'order_quarter', 'revenue_calc',
          'discount_pct', 'contact_full', 'full_address']
for c in nuevas:
    print(f"   ✚ {c}")

print("\n🏠 Ejemplo de full_address:")
print(df['full_address'].iloc[0])

print("\n📋 Vista del dataset enriquecido (primeras 3 filas):")
display(df[['ordernumber','orderdate','customername','productline',
            'productcode','full_address','sales','status','dealsize','dealsize_num']].head(3))

---
## 📄 PASO 7 — Convertir Filas del Dataset → Documentos RAG

Este es el **paso clave** para datos tabulares.  
Cada fila se convierte en un **texto descriptivo rico** que incluye:
- Ubicación exacta (dirección completa, ciudad, país, territorio)
- Producto (línea, código, MSRP, descuento calculado)
- Transacción (cantidad, precio, total, deal size y su valor ordinal)
- Estado y fechas

El retriever buscará semánticamente en estos textos para responder  
preguntas como: *"¿Dónde está el producto S10_1678?"*

In [ ]:
# ═══ CONVERTIR CADA FILA EN UN DOCUMENTO TEXTUAL PARA RAG ════════

def row_to_document(row):
    """
    Convierte una fila del DataFrame normalizado en un Documento RAG.
    Usa los nombres de columna en snake_case del dataset normalizado.
    """
    orderdate_str = row['orderdate'].strftime('%d/%m/%Y') if pd.notna(row['orderdate']) else 'N/A'

    text = (
        f"ORDEN #{row['ordernumber']} | FECHA: {orderdate_str} | "
        f"ESTADO: {row['status']} | DEAL: {row['dealsize']} (nivel {row['dealsize_num']})\n"
        f"CLIENTE: {row['customername']} | CONTACTO: {row['contact_full']} | TEL: {row['phone']}\n"
        f"UBICACIÓN: {row['full_address']} | CÓDIGO POSTAL: {row['postalcode']} | TERRITORIO: {row['territory']}\n"
        f"PRODUCTO: {row['productline']} | CÓDIGO: {row['productcode']} | MSRP: ${row['msrp']}\n"
        f"PRECIO UNITARIO: ${row['priceeach']} | DESCUENTO: {row['discount_pct']}% | "
        f"CANTIDAD: {row['quantityordered']} unidades\n"
        f"VENTA TOTAL: ${row['sales']} | INGRESOS CALCULADOS: ${row['revenue_calc']}\n"
        f"AÑO: {row['order_year']} | MES: {row['order_month']} | TRIMESTRE: Q{row['order_quarter']} | "
        f"LÍNEA DE ORDEN: {row['orderlinenumber']}"
    )

    metadata = {
        "ordernumber"  : str(row['ordernumber']),
        "customer"     : row['customername'],
        "country"      : row['country'],
        "city"         : row['city'],
        "territory"    : str(row['territory']),
        "address"      : row['full_address'],
        "productline"  : row['productline'],
        "productcode"  : row['productcode'],
        "status"       : row['status'],
        "dealsize"     : row['dealsize'],
        "dealsize_num" : int(row['dealsize_num']),
        "year"         : int(row['order_year']) if pd.notna(row['order_year']) else 0,
        "sales"        : float(row['sales']),
        "discount_pct" : float(row['discount_pct']),
        "quantity"     : int(row['quantityordered']),
    }
    return Document(page_content=text, metadata=metadata)

# Convertir todas las filas
print("⏳ Convirtiendo 2823 filas a documentos RAG...")
documents = [row_to_document(row) for _, row in df.iterrows()]

print(f"\n✅ {len(documents):,} documentos creados")
print("\n" + "─"*70)
print("📄 EJEMPLO DE DOCUMENTO GENERADO:")
print("─"*70)
print(documents[0].page_content)
print("─"*70)
print("\n🏷️  METADATA DEL DOCUMENTO:")
for k, v in documents[0].metadata.items():
    print(f"   {k:<14}: {v}")

---
## 🔍 PASO 8 — Embeddings + Índice Vectorial FAISS

### ¿Qué son los embeddings?
Son representaciones numéricas de texto (vectores de ~384 dimensiones).  
Textos semánticamente similares tienen vectores cercanos en el espacio.

### Pipeline:
```
"¿Dónde está el producto S10_1678?"
              │
              ▼  [Embedding model]
    [0.12, -0.34, 0.87, ...]   ← vector de la pregunta
              │
              ▼  [FAISS búsqueda de vecinos más cercanos]
    Documentos del corpus con vectores similares
    → Órdenes de S10_1678 con sus direcciones completas
```

In [ ]:
# ═══ EMBEDDINGS + CONSTRUCCIÓN DEL ÍNDICE FAISS ═════════════════

print("⏳ Cargando modelo de embeddings multilingüe...")
print("   Modelo: paraphrase-multilingual-MiniLM-L12-v2")
print("   (soporta español, inglés y otros — ideal para este dataset)\n")

embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Modelo de embeddings cargado (local, sin API Key)")
print("\n⏳ Construyendo índice FAISS sobre 2823 documentos...")
print("   (puede tardar 1-2 minutos)\n")

vectorstore = FAISS.from_documents(documents, embeddings_model)
vectorstore.save_local("faiss_sales_index")

print(f"✅ Índice FAISS construido y guardado")
print(f"   Vectores indexados : {vectorstore.index.ntotal:,}")
print(f"   Dimensión          : {vectorstore.index.d}")
print(f"   Guardado en        : faiss_sales_index/")

# ── Verificar con búsquedas de prueba ────────────────────────────
print("\n🔎 TESTS DE BÚSQUEDA SEMÁNTICA:")
tests = [
    ("producto S10_1678 en USA", 2),
    ("órdenes canceladas classic cars", 2),
    ("cliente en Francia motorcycles", 2),
]
for query, k in tests:
    results = vectorstore.similarity_search(query, k=k)
    print(f"\n  Query: '{query}'")
    for i, doc in enumerate(results, 1):
        m = doc.metadata
        print(f"    [{i}] Orden #{m['ordernumber']} | {m['customer']} | "
              f"{m['productline']} | {m['city']}, {m['country']}")

---
## 🦾 PASO 9 — Configurar el Agente RAG con Mistral

### Arquitectura del Agente

| Tool | ¿Qué hace? |
|---|---|
| `buscar_en_corpus` | Búsqueda semántica libre en todo el dataset |
| `localizar_producto` | Dice exactamente dónde está un producto (ciudad, país, dirección) |
| `filtrar_por_pais` | Filtra órdenes por país |
| `filtrar_por_producto` | Filtra por línea de producto |
| `resumen_estadistico` | Estadísticas numéricas del dataset |

In [ ]:
# ═══ CONFIGURAR MISTRAL LLM ══════════════════════════════════════

llm = ChatMistralAI(
    model="mistral-small-latest",
    mistral_api_key=MISTRAL_API_KEY,
    temperature=0.05,
    max_tokens=1500
)
print("✅ Mistral LLM configurado (mistral-small-latest)")

# ═══ RETRIEVER BASE ═══════════════════════════════════════════════
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

# ═══ PROMPT PERSONALIZADO ════════════════════════════════════════
SYSTEM_PROMPT = """Sos un asistente experto en análisis del corpus de ventas.
Tu trabajo es responder preguntas usando ÚNICAMENTE la información del contexto proporcionado.

REGLAS ESTRICTAS:
1. Basate SOLO en los datos del contexto. No inventes ni asumas nada.
2. Si la pregunta es sobre UBICACIÓN, indicá siempre: dirección, ciudad, país y territorio.
3. Si la pregunta es sobre un PRODUCTO, indicá: código, línea, MSRP, descuento y dónde se vendió.
4. Si hay MÚLTIPLES resultados, listalos en formato claro.
5. Si no encontrás la información, decí: "No encontré esa información en el corpus."
6. Citá el número de orden cuando sea relevante.
7. Respondé en español, de forma concisa y estructurada.

CONTEXTO DEL CORPUS:
{context}

PREGUNTA: {question}

RESPUESTA ESTRUCTURADA:"""

rag_prompt = PromptTemplate(
    template=SYSTEM_PROMPT,
    input_variables=["context", "question"]
)

# ═══ CADENA RAG BASE ══════════════════════════════════════════════
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": rag_prompt},
    return_source_documents=True
)
print("✅ Cadena RAG base configurada")

# ═══ TOOLS ESPECIALIZADAS DEL AGENTE ═════════════════════════════

def buscar_en_corpus(query: str) -> str:
    """Búsqueda semántica libre en el corpus de ventas."""
    result = qa_chain.invoke({"query": query})
    return result["result"]

def localizar_producto(producto: str) -> str:
    """Localiza exactamente dónde está/fue vendido un producto. Usa columnas snake_case."""
    query_text = f"ubicación dirección ciudad país de {producto}"
    docs = vectorstore.similarity_search(query_text, k=10)

    ubicaciones = []
    vistos = set()
    for doc in docs:
        m = doc.metadata
        key = (m['productcode'], m['customer'])
        if key not in vistos:
            vistos.add(key)
            if (producto.upper() in m['productcode'].upper() or
                producto.lower() in m['productline'].lower() or
                producto.lower() in m['customer'].lower()):
                ubicaciones.append(m)

    if not ubicaciones:
        result = qa_chain.invoke({"query": f"¿Dónde está el producto {producto}? Dame dirección, ciudad y país."})
        return result["result"]

    respuesta = f"📍 UBICACIONES DE '{producto.upper()}':\n"
    respuesta += "─" * 60 + "\n"
    for u in ubicaciones[:8]:
        respuesta += (
            f"  Orden #{u['ordernumber']} | {u['productcode']} ({u['productline']})\n"
            f"  Cliente  : {u['customer']}\n"
            f"  Dirección: {u['address']}\n"
            f"  País     : {u['country']} | Territorio: {u['territory']}\n"
            f"  Venta    : ${u['sales']} | Estado: {u['status']}\n"
            f"  {'─'*55}\n"
        )
    return respuesta

def filtrar_por_pais(pais: str) -> str:
    """Filtra y resume órdenes de un país específico. Usa columnas snake_case."""
    subset = df[df['country'].str.contains(pais.strip(), case=False, na=False)]

    if subset.empty:
        return f"No se encontraron órdenes para el país: {pais}"

    resumen = (
        f"📊 ÓRDENES DE {pais.upper()}:\n"
        f"  Total órdenes    : {len(subset):,}\n"
        f"  Ventas totales   : ${subset['sales'].sum():,.2f}\n"
        f"  Venta promedio   : ${subset['sales'].mean():,.2f}\n"
        f"  Clientes únicos  : {subset['customername'].nunique()}\n"
        f"  Productos        : {', '.join(subset['productline'].unique())}\n"
        f"  Estados          : {', '.join(subset['status'].unique())}\n"
        f"\n  Top clientes:\n"
    )
    top = subset.groupby('customername')['sales'].sum().nlargest(5)
    for cust, sales in top.items():
        resumen += f"    • {cust}: ${sales:,.2f}\n"
    return resumen

def filtrar_por_producto(linea: str) -> str:
    """Filtra y resume órdenes por línea de producto. Usa columnas snake_case."""
    subset = df[df['productline'].str.contains(linea.strip(), case=False, na=False)]

    if subset.empty:
        return f"No se encontraron órdenes para la línea: {linea}"

    resumen = (
        f"🚗 LÍNEA DE PRODUCTO: {linea.upper()}\n"
        f"  Total órdenes    : {len(subset):,}\n"
        f"  Ventas totales   : ${subset['sales'].sum():,.2f}\n"
        f"  Precio promedio  : ${subset['priceeach'].mean():,.2f}\n"
        f"  Descuento prom.  : {subset['discount_pct'].mean():.1f}%\n"
        f"  MSRP promedio    : ${subset['msrp'].mean():,.2f}\n"
        f"  Deal size prom.  : {subset['dealsize_num'].mean():.2f} (1=Small, 2=Medium, 3=Large)\n"
        f"  Países activos   : {', '.join(subset['country'].unique()[:8])}\n"
        f"  Deal sizes       : {subset['dealsize'].value_counts().to_dict()}\n"
        f"\n  Productos más vendidos (códigos):\n"
    )
    top_prod = subset.groupby('productcode')['sales'].sum().nlargest(5)
    for code, sales in top_prod.items():
        resumen += f"    • {code}: ${sales:,.2f}\n"
    return resumen

def resumen_estadistico(parametro: str) -> str:
    """Devuelve estadísticas generales del dataset normalizado."""
    total_ventas = df['sales'].sum()
    resumen = (
        f"📈 RESUMEN ESTADÍSTICO DEL DATASET NORMALIZADO\n"
        f"{'─'*50}\n"
        f"  Total filas           : {len(df):,}\n"
        f"  Columnas              : {len(df.columns)}\n"
        f"  Ventas totales        : ${total_ventas:,.2f}\n"
        f"  Venta máxima          : ${df['sales'].max():,.2f}\n"
        f"  Venta mínima          : ${df['sales'].min():,.2f}\n"
        f"  Venta promedio        : ${df['sales'].mean():,.2f}\n"
        f"  Descuento prom. (%)   : {df['discount_pct'].mean():.1f}%\n"
        f"  Años en el dataset    : {sorted(df['order_year'].dropna().unique().astype(int).tolist())}\n"
        f"  Países               : {df['country'].nunique()} únicos\n"
        f"  Territorios          : {', '.join(df['territory'].dropna().unique())}\n"
        f"  Clientes             : {df['customername'].nunique()} únicos\n"
        f"  Líneas de producto   : {', '.join(df['productline'].unique())}\n"
        f"  Deal sizes           : {df['dealsize'].value_counts().to_dict()}\n"
        f"  Deal size prom.      : {df['dealsize_num'].mean():.2f} (1=S, 2=M, 3=L)\n"
        f"  Estados de órdenes   : {df['status'].value_counts().to_dict()}\n"
    )
    return resumen

# ═══ DEFINIR TOOLS PARA EL AGENTE ════════════════════════════════
tools = [
    Tool(
        name="buscar_en_corpus",
        func=buscar_en_corpus,
        description=(
            "Busca información en el corpus de ventas con lenguaje natural. "
            "Úsala para preguntas generales sobre órdenes, clientes, fechas o ventas."
        )
    ),
    Tool(
        name="localizar_producto",
        func=localizar_producto,
        description=(
            "Localiza dónde está un producto en el corpus: devuelve dirección completa, "
            "ciudad, país y territorio. Input: código de producto o nombre de línea."
        )
    ),
    Tool(
        name="filtrar_por_pais",
        func=filtrar_por_pais,
        description=(
            "Filtra y resume todas las órdenes de un país específico. "
            "Input: nombre del país en inglés o español."
        )
    ),
    Tool(
        name="filtrar_por_producto",
        func=filtrar_por_producto,
        description=(
            "Filtra y resume ventas por línea de producto: Classic Cars, Motorcycles, "
            "Vintage Cars, Trucks and Buses, Planes, Ships, Trains."
        )
    ),
    Tool(
        name="resumen_estadistico",
        func=resumen_estadistico,
        description=(
            "Devuelve estadísticas generales del dataset completo: totales, promedios, "
            "distribuciones. Input: cualquier texto (ej: 'general')."
        )
    ),
]

# ═══ INICIALIZAR EL AGENTE ════════════════════════════════════════
agente = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=4,
    handle_parsing_errors=True
)

print("\n✅ AGENTE RAG configurado con 5 herramientas especializadas")
print("   • buscar_en_corpus     — búsqueda semántica libre")
print("   • localizar_producto   — ubicación exacta de productos")
print("   • filtrar_por_pais     — análisis por país")
print("   • filtrar_por_producto — análisis por línea de producto")
print("   • resumen_estadistico  — estadísticas generales")

---
## 🧪 PASO 10 — Probar el Agente con Preguntas de Ejemplo

In [ ]:
# ─── Función helper para mostrar respuestas ──────────────────────
def preguntar(pregunta: str, usar_agente: bool = True):
    print("\n" + "╔" + "═"*66 + "╗")
    print(f"║  ❓ {pregunta[:62]:<62} ║")
    print("╚" + "═"*66 + "╝")

    if usar_agente:
        try:
            respuesta = agente.run(pregunta)
        except Exception as e:
            respuesta = buscar_en_corpus(pregunta)
    else:
        resultado = qa_chain.invoke({"query": pregunta})
        respuesta = resultado["result"]

    print("\n🤖 RESPUESTA DEL AGENTE RAG:")
    print("─" * 68)
    print(respuesta)
    print("─" * 68)
    return respuesta

In [ ]:
# TEST 1: Localización de producto
preguntar("¿Dónde está el producto S10_1678? Dame la dirección exacta, ciudad y país.")

In [ ]:
# TEST 2: Filtro por país
preguntar("¿Qué órdenes hay de clientes en Francia? ¿Cuánto es el total de ventas?")

In [ ]:
# TEST 3: Estado de órdenes
preguntar("¿Hay órdenes canceladas? ¿De qué productos son y en qué países están?")

In [ ]:
# TEST 4: Línea de producto + ubicación
preguntar("¿En qué países se venden motocicletas? ¿Cuál es la orden de mayor venta?")

In [ ]:
# TEST 5: Estadísticas (incluyendo dealsize_num del dataset normalizado)
preguntar("Dame un resumen estadístico completo del dataset de ventas.")

In [ ]:
# TEST 6: Ubicación de cliente
preguntar("¿En qué ciudad y dirección está el cliente Land Of Toys?")

---
## 💬 PASO 11 — Bot Interactivo — Tu turno de preguntar

### Ejemplos de preguntas que podés hacer:
- `¿Dónde está el producto S18_4409?`
- `¿Qué clientes hay en Australia?`
- `¿Cuál es el deal más grande del 2004?`
- `¿Hay órdenes en proceso? ¿Cuáles son y dónde están?`
- `¿Qué línea de producto tiene más ventas?`
- `Dame los Classic Cars vendidos en España`
- `¿Qué territorio tiene mayor descuento promedio?` ← nueva gracias a discount_pct
- `¿Cuántos Large deals hay por año?` ← usa dealsize_num del dataset normalizado

Escribí `salir` para terminar.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🤖 BOT RAG INTERACTIVO — SALES DATASET NORMALIZADO
# ══════════════════════════════════════════════════════════════
print("╔══════════════════════════════════════════════════════════╗")
print("║   🤖 RAG BOT — Sales Dataset │ Powered by Mistral AI   ║")
print("║   📊 Corpus: 2823 órdenes │ 7 líneas │ 19 países        ║")
print("║   ✅ Dataset: normalizado (snake_case, sin nulos)        ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Herramientas disponibles:                               ║")
print("║   🔍 Búsqueda semántica en todo el corpus               ║")
print("║   📍 Localización exacta de productos                   ║")
print("║   🌎 Filtro por país                                    ║")
print("║   🚗 Filtro por línea de producto                       ║")
print("║   📈 Estadísticas y resúmenes                           ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Escribí 'salir' para terminar │ 'ayuda' para ejemplos  ║")
print("╚══════════════════════════════════════════════════════════╝")

EJEMPLOS = [
    "¿Dónde está el producto S10_1678?",
    "¿Qué clientes hay en Australia?",
    "¿Cuál es el deal más grande del 2004?",
    "Mostrame órdenes en proceso",
    "Dame un resumen estadístico",
    "¿Qué Classic Cars se vendieron en España?",
    "¿Qué territorio tiene mayor descuento promedio?",
    "¿Cuántos Large deals hay por año?",
]

while True:
    print()
    user_input = input("💬 Tu pregunta: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ['salir', 'exit', 'quit', 'q']:
        print("\n👋 ¡Hasta luego! Bot finalizado.")
        break

    if user_input.lower() == 'ayuda':
        print("\n💡 Ejemplos de preguntas:")
        for i, ej in enumerate(EJEMPLOS, 1):
            print(f"   {i}. {ej}")
        continue

    print("\n⏳ Procesando con el agente RAG...")
    try:
        respuesta = agente.run(user_input)
        print("\n🤖 RESPUESTA:")
        print("─" * 65)
        print(respuesta)
        print("─" * 65)
    except Exception as e:
        try:
            resultado = qa_chain.invoke({"query": user_input})
            print("\n🤖 RESPUESTA (RAG directo):")
            print("─" * 65)
            print(resultado["result"])
            print("─" * 65)
        except Exception as e2:
            print(f"❌ Error: {e2}")

---
## 📊 Apéndice — Análisis Pandas Complementario

Consultas directas al DataFrame normalizado para validar respuestas del bot o hacer análisis más profundos.

In [ ]:
# ─── Top 10 órdenes por monto de venta ───────────────────────────
print("🏆 TOP 10 ÓRDENES POR MONTO DE VENTA")
top10 = df.nlargest(10, 'sales')[
    ['ordernumber','customername','productline','productcode',
     'sales','status','city','country','full_address']
].reset_index(drop=True)
display(top10)

In [ ]:
# ─── Ventas por país y línea de producto (pivot) ──────────────────
print("📊 VENTAS POR PAÍS Y LÍNEA DE PRODUCTO")
pivot = df.pivot_table(
    values='sales',
    index='country',
    columns='productline',
    aggfunc='sum',
    fill_value=0
).round(0).astype(int)
pivot['TOTAL'] = pivot.sum(axis=1)
pivot = pivot.sort_values('TOTAL', ascending=False)
display(pivot.head(15))

In [ ]:
# ─── Deal size por territorio (usando dealsize_num del dataset normalizado) ──
print("🌎 DEAL SIZE PROMEDIO POR TERRITORIO (1=Small, 2=Medium, 3=Large)")
deal_terr = df.groupby('territory').agg(
    N_Ordenes=('ordernumber','count'),
    Venta_Total=('sales','sum'),
    DealSize_Prom=('dealsize_num','mean'),
    Descuento_Prom=('discount_pct','mean'),
).round(2).sort_values('Venta_Total', ascending=False)
display(deal_terr)

In [ ]:
# ─── Productos únicos con su ubicación de venta ──────────────────
print("📍 CADA CÓDIGO DE PRODUCTO → CIUDADES DONDE FUE VENDIDO")
prod_loc = (
    df.groupby(['productcode', 'productline'])
    .apply(lambda x: ', '.join(sorted(x['city'].unique()[:5])))
    .reset_index()
    .rename(columns={0: 'Ciudades'})
)
display(prod_loc.head(20))

In [ ]:
# ─── Órdenes por estado con totales ──────────────────────────────
print("📋 ÓRDENES POR ESTADO")
status_summary = df.groupby('status').agg(
    N_Ordenes=('ordernumber','count'),
    Venta_Total=('sales','sum'),
    Venta_Promedio=('sales','mean'),
    Paises=('country', lambda x: x.nunique()),
).round(2)
display(status_summary)